In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.interpolate import RegularGridInterpolator
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

from itertools import product
import matplotlib.colors as mcolors
from wrf_io import postproc

In [3]:
casenames = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/sweep_names.npy')
U_loc   = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/U_loc.npy')
dir_loc = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/dir_loc.npy')
U_disk   = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/U_disk.npy')
dir_disk = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/dir_disk.npy')
Uhub    = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/Uhub.npy')
wrf_tsr = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/wrf_tsr.npy')
wrf_omg = np.load('/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/wrf_omg.npy')
Nelm    = np.load("/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/rotor_dims_10MW.npz")['Nelm']
Nsct    = np.load("/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/rotor_dims_10MW.npz")['Nsct']
rOverR  = np.load("/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/rotor_dims_10MW.npz")['rOverR']
R       = np.load("/scratch/09909/smata/induction_modeling/madsen_modeling/rotorAvg_10MW/processedData/rotor_dims_10MW.npz")['R']

In [4]:
veers = np.array([-0.2, -0.1, -0.05, -0.025, 0.0, 0.025, 0.05, 0.1, 0.2])

In [58]:
from MITRotor import BEM, BEMGeometry, IEA10MW, NoTipLoss, WRFLESAerodynamics, NoTangentialInduction, ConstantInduction

In [59]:
# LES Baseline

rotor = IEA10MW()

base_les_ind = np.zeros_like(casenames, dtype='float')
base_les_cot = np.zeros_like(casenames, dtype='float')
base_les_thr = np.zeros_like(casenames, dtype='float')
base_les_cop = np.zeros_like(casenames, dtype='float')
base_les_pow = np.zeros_like(casenames, dtype='float')
iters   = np.zeros_like(casenames, dtype='float')

for count in range(len(base_les_ind)):

    bem = BEM(rotor=rotor, geometry=BEMGeometry(Nr=Nelm,Ntheta=Nsct,R=rotor.R, Rhub = rotor.hub_radius), aerodynamic_model=WRFLESAerodynamics(), tiploss_model=NoTipLoss(), momentum_model=ConstantInduction(a=0), tangential_induction_model=NoTangentialInduction())

    sol = bem(0, wrf_tsr[count], 0, v_inf = Uhub[count], U = U_disk[:,:,count]/Uhub[count], wdir=dir_disk[:,:,count], veer = veers[count])

    base_les_cot[count] = sol.Ct()
    base_les_thr[count] = sol.thrust()/1e3
    base_les_pow[count] = sol.power()/1e6
    base_les_cop[count] = sol.Cp()
    base_les_ind[count] = sol.a()
    iters[count]   = sol.niter

iters

array([1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [79]:
from MITRotor import BEM, BEMGeometry, IEA10MW, NoTipLoss, WRFLESAerodynamics, NoTangentialInduction, ClassicalMomentum

In [80]:
# 1D Momentum Baseline

rotor = IEA10MW()

base_1D_ind = np.zeros_like(casenames, dtype='float')
base_1D_cot = np.zeros_like(casenames, dtype='float')
base_1D_thr = np.zeros_like(casenames, dtype='float')
base_1D_cop = np.zeros_like(casenames, dtype='float')
base_1D_pow = np.zeros_like(casenames, dtype='float')
iters   = np.zeros_like(casenames, dtype='float')

for count in range(len(base_1D_ind)):

    bem = BEM(rotor=rotor, geometry=BEMGeometry(Nr=Nelm,Ntheta=Nsct,R=rotor.R, Rhub = rotor.hub_radius), aerodynamic_model=WRFLESAerodynamics(), tiploss_model=NoTipLoss(), momentum_model=ClassicalMomentum(), tangential_induction_model=NoTangentialInduction())

    sol = bem(0, wrf_tsr[count], 0, v_inf = Uhub[count], U = U_loc[:,:,count]/Uhub[count], wdir=dir_loc[:,:,count], veer = veers[count])

    base_1D_cot[count] = sol.Ct()
    base_1D_thr[count] = sol.thrust()/1e3
    base_1D_pow[count] = sol.power()/1e6
    base_1D_cop[count] = sol.Cp()
    base_1D_ind[count] = sol.a()
    iters[count]   = sol.niter

iters

array([7., 7., 7., 7., 7., 7., 7., 7., 8.])

In [81]:
from MITRotor import BEM, BEMGeometry, IEA10MW, NoTipLoss, WRFLESAerodynamics, NoTangentialInduction, MadsenMomentum

In [82]:
# Madsen Baseline

rotor = IEA10MW()

base_mad_ind = np.zeros_like(casenames, dtype='float')
base_mad_cot = np.zeros_like(casenames, dtype='float')
base_mad_thr = np.zeros_like(casenames, dtype='float')
base_mad_cop = np.zeros_like(casenames, dtype='float')
base_mad_pow = np.zeros_like(casenames, dtype='float')
iters   = np.zeros_like(casenames, dtype='float')

for count in range(len(base_mad_ind)):

    bem = BEM(rotor=rotor, geometry=BEMGeometry(Nr=Nelm,Ntheta=Nsct,R=rotor.R, Rhub = rotor.hub_radius), aerodynamic_model=WRFLESAerodynamics(), tiploss_model=NoTipLoss(), momentum_model=MadsenMomentum(averaging='rotor'), tangential_induction_model=NoTangentialInduction())

    sol = bem(0, wrf_tsr[count], 0, v_inf = Uhub[count], U = U_loc[:,:,count]/Uhub[count], wdir=dir_loc[:,:,count], veer = veers[count])

    base_mad_cot[count] = sol.Ct()
    base_mad_thr[count] = sol.thrust()/1e3
    base_mad_pow[count] = sol.power()/1e6
    base_mad_cop[count] = sol.Cp()
    base_mad_ind[count] = sol.a()
    iters[count]   = sol.niter
    
iters

array([7., 7., 7., 7., 7., 7., 7., 7., 8.])

In [5]:
from MITRotor import BEM, BEMGeometry, IEA10MW, NoTipLoss, WRFLESAerodynamics, NoTangentialInduction, GP_Rotor

In [28]:
rotor = IEA10MW()

mit_ind = np.zeros_like(casenames, dtype='float')
mit_cot = np.zeros_like(casenames, dtype='float')
mit_thr = np.zeros_like(casenames, dtype='float')
mit_cop = np.zeros_like(casenames, dtype='float')
mit_pow = np.zeros_like(casenames, dtype='float')
iters   = np.zeros_like(casenames, dtype='float')
mit_w   = np.zeros_like(casenames, dtype='float')
mit_cax = np.zeros_like(casenames, dtype='float')
mit_vax = np.zeros_like(casenames, dtype='float')
mit_vtn = np.zeros_like(casenames, dtype='float')

# for count in range(len(mit_ind)):
for count in [3]:

    bem = BEM(rotor=rotor, geometry=BEMGeometry(Nr=Nelm,Ntheta=Nsct,R=rotor.R, Rhub = rotor.hub_radius), aerodynamic_model=WRFLESAerodynamics(), tiploss_model=NoTipLoss(), momentum_model=GP_Rotor(veer=veers[count]), tangential_induction_model=NoTangentialInduction())

    sol = bem(0, wrf_tsr[count], 0, v_inf = Uhub[count], U = U_loc[:,:,count]/Uhub[count], wdir=dir_loc[:,:,count], veer = veers[count])

    mit_cot[count] = sol.Ct()
    mit_thr[count] = sol.thrust()/1e3
    mit_pow[count] = sol.power()/1e6
    mit_cop[count] = sol.Cp()
    mit_ind[count] = sol.a()
    iters[count]   = sol.niter
    mit_w[count]   = sol.W()
    mit_cax[count] = sol.Cx()
    mit_vax[count] = sol.Vax()
    mit_vtn[count] = sol.Vtan()
    
iters

CT:0.6495995178344071
CT:[-5.48839802]
sheer:[0.]
veer:[-0.2300895]
a:[0.29149945]
CT:0.6825765935775452
CT:[-0.6844641]
sheer:[0.]
veer:[-0.2300895]
a:[0.25036]
CT:0.7128418077360041
CT:[3.72441951]
sheer:[0.]
veer:[-0.2300895]
a:[0.20780497]
CT:0.7419167750798712
CT:[7.95991397]
sheer:[0.]
veer:[-0.2300895]
a:[0.17604564]
CT:0.7648333838743137
CT:[11.29828985]
sheer:[0.]
veer:[-0.2300895]
a:[0.1631086]
CT:0.7767274692227101
CT:[13.03096016]
sheer:[0.]
veer:[-0.2300895]
a:[0.16106603]
CT:0.7806548853531876
CT:[13.60308632]
sheer:[0.]
veer:[-0.2300895]
a:[0.1610581]
CT:0.7816408886723049
CT:[13.7467223]
sheer:[0.]
veer:[-0.2300895]
a:[0.16110556]
CT:0.7818645154454847
CT:[13.77929912]
sheer:[0.]
veer:[-0.2300895]
a:[0.16111903]
CT:0.7819139235138604
CT:[13.78649664]
sheer:[0.]
veer:[-0.2300895]
a:[0.16112214]


array([0., 0., 0., 9., 0., 0., 0., 0., 0.])